# GemmaFit v5 E2B LiteRT-LM conversion matrix

This notebook converts the already-merged Hugging Face model into LiteRT-LM artifacts. It does not train, merge LoRA, or import Unsloth.

Default source model:
`/content/drive/MyDrive/GemmaFit_train/trained_outputs/merged_fp16/google_gemma_4_E2B_it_gemmafit_v5_2_e2b_completion_only_repair3_contract_smoke`

Primary target:
`gemmafit-v5-e2b-evidence-router.litertlm`

Conversion policy:
- Start with `cache_length=2048`, `prefill_lengths=[1024]`, `dynamic_wi8_afp32`.
- If export or Pixel init fails, keep `cache_length=2048` and reduce prefill to `[512]` first.
- Use `cache_length=1024` only after the app prompt is compressed below 1024 tokens.
- Use `wi4` only if wi8 packaging or runtime memory fails.

In [ ]:
# Cell 1: Drive, paths, matrix config, durable state.
from __future__ import annotations

import datetime as dt
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path
from typing import Any

try:
    from google.colab import drive
except Exception as exc:  # Allows local syntax checks outside Colab.
    drive = None
    print(f"google.colab import skipped: {exc}")

if drive is not None:
    if Path('/content/drive/MyDrive').exists():
        print('Drive already mounted.')
    else:
        drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/GemmaFit_train/trained_outputs')
RUN_NAME = 'google_gemma_4_E2B_it_gemmafit_v5_2_e2b_completion_only_repair3_contract_smoke'
ARTIFACT_PREFIX = 'gemmafit-v5-e2b-evidence-router'

MERGED_PATH = Path(os.environ.get(
    'MERGED_PATH',
    str(DRIVE_ROOT / 'merged_fp16' / RUN_NAME),
))

EXPORT_MATRIX_DIR = DRIVE_ROOT / 'litert_export_v5_e2b_matrix'
LOCAL_WORK_ROOT = Path('/content/gemmafit_v5_e2b_litert_matrix_work')
LITERTLM_FINAL = DRIVE_ROOT / f'{ARTIFACT_PREFIX}.litertlm'
RUN_STATE = DRIVE_ROOT / f'RUN_STATE_{RUN_NAME}_litert_matrix.json'
RUN_EVENTS = DRIVE_ROOT / f'RUN_EVENTS_{RUN_NAME}_litert_matrix.jsonl'
DISCONNECT_POINTS = DRIVE_ROOT / f'DISCONNECT_POINTS_{RUN_NAME}_litert_matrix.jsonl'
MATRIX_SUMMARY = DRIVE_ROOT / f'LITERT_MATRIX_{RUN_NAME}.json'

FORCE_EXPORT = os.environ.get('FORCE_EXPORT', '0') == '1'
FORCE_FINAL = os.environ.get('FORCE_FINAL', '0') == '1'
STOP_AFTER_FIRST_SUCCESS = os.environ.get('STOP_AFTER_FIRST_SUCCESS', '0') == '1'
RUN_VARIANTS = [
    name.strip()
    for name in os.environ.get(
        'RUN_VARIANTS',
        'wi8_ctx2048_prefill1024,wi8_ctx2048_prefill512',
    ).split(',')
    if name.strip()
]

VARIANTS = [
    {
        'name': 'wi8_ctx2048_prefill1024',
        'quantization_recipe': 'dynamic_wi8_afp32',
        'cache_length': 2048,
        'prefill_lengths': [1024],
        'note': 'official-aligned Gemma 4 E2B target; use this first',
    },
    {
        'name': 'wi8_ctx2048_prefill512',
        'quantization_recipe': 'dynamic_wi8_afp32',
        'cache_length': 2048,
        'prefill_lengths': [512],
        'note': 'same context, lighter prefill graph; first fallback',
    },
    {
        'name': 'wi8_ctx1024_prefill512',
        'quantization_recipe': 'dynamic_wi8_afp32',
        'cache_length': 1024,
        'prefill_lengths': [512],
        'note': 'only if app prompt is compressed below 1024 tokens',
    },
    {
        'name': 'wi4_ctx2048_prefill512',
        'quantization_recipe': 'dynamic_wi4_afp32',
        'cache_length': 2048,
        'prefill_lengths': [512],
        'note': 'smaller weights; try if wi8 package/runtime memory fails',
    },
]
VARIANT_BY_NAME = {variant['name']: variant for variant in VARIANTS}

unknown_variants = [name for name in RUN_VARIANTS if name not in VARIANT_BY_NAME]
if unknown_variants:
    raise ValueError(f'Unknown RUN_VARIANTS entries: {unknown_variants}')

EXPORT_MATRIX_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_WORK_ROOT.mkdir(parents=True, exist_ok=True)


def now_iso() -> str:
    return dt.datetime.now(dt.timezone.utc).isoformat()


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')
    tmp.replace(path)


def append_jsonl(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(payload, sort_keys=True) + '\n')


def record_event(event: str, **payload: Any) -> None:
    row = {'ts': now_iso(), 'event': event, **payload}
    append_jsonl(RUN_EVENTS, row)
    print(json.dumps(row, sort_keys=True))


def file_sha256(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def valid_merged_hf_model(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False
    if not (path / 'config.json').exists():
        return False
    weight_files = list(path.glob('*.safetensors')) + list(path.glob('pytorch_model*.bin'))
    return any(item.stat().st_size > 0 for item in weight_files)


def resolve_merged_path() -> Path:
    if valid_merged_hf_model(MERGED_PATH):
        return MERGED_PATH
    parent = DRIVE_ROOT / 'merged_fp16'
    candidates = sorted(parent.glob('*v5*e2b*completion_only_repair3*')) if parent.exists() else []
    for candidate in candidates:
        if valid_merged_hf_model(candidate):
            return candidate
    raise FileNotFoundError(
        'Merged Hugging Face model not found. Set MERGED_PATH or confirm the Drive export exists: '
        f'{MERGED_PATH}'
    )


def prefill_arg(lengths: list[int]) -> str:
    return '[' + ','.join(str(item) for item in lengths) + ']'


def variant_artifact_path(name: str) -> Path:
    return EXPORT_MATRIX_DIR / f'{ARTIFACT_PREFIX}-{name}.litertlm'


def variant_export_dir(name: str) -> Path:
    return EXPORT_MATRIX_DIR / name


def variant_log_path(name: str) -> Path:
    return EXPORT_MATRIX_DIR / f'{ARTIFACT_PREFIX}-{name}-export.log'


def bundle_path(name: str) -> Path:
    return EXPORT_MATRIX_DIR / f'{ARTIFACT_PREFIX}-{name}-local-artifacts.zip'


def find_litertlm(root: Path) -> Path:
    matches = [path for path in root.rglob('*.litertlm') if path.is_file() and path.stat().st_size > 0]
    if not matches:
        raise FileNotFoundError(f'No non-empty .litertlm file found under {root}')
    return max(matches, key=lambda path: path.stat().st_size)

MERGED_PATH = resolve_merged_path()
write_json(RUN_STATE, {
    'run_name': RUN_NAME,
    'artifact_prefix': ARTIFACT_PREFIX,
    'merged_path': str(MERGED_PATH),
    'run_variants': RUN_VARIANTS,
    'force_export': FORCE_EXPORT,
    'force_final': FORCE_FINAL,
    'stop_after_first_success': STOP_AFTER_FIRST_SUCCESS,
    'litertlm_final': str(LITERTLM_FINAL),
    'export_matrix_dir': str(EXPORT_MATRIX_DIR),
    'state_ts': now_iso(),
})
record_event('matrix_config_ready', merged_path=str(MERGED_PATH), variants=RUN_VARIANTS)
print(f'MERGED_PATH={MERGED_PATH}')
print(f'RUN_VARIANTS={RUN_VARIANTS}')

## Runtime dependency install

Run this cell once per fresh Colab runtime, then restart the runtime if Torch or torchao changed. After restart, rerun Cell 1 and continue from the exporter verification cell.

In [ ]:
# Cell 2: Install LiteRT-LM exporter dependencies.
# If this upgrades torch/torchao, restart the Colab runtime and resume from Cell 1.
INSTALL_DEPS = os.environ.get('INSTALL_DEPS', '1') == '1'
AUTO_RESTART_AFTER_INSTALL = os.environ.get('AUTO_RESTART_AFTER_INSTALL', '0') == '1'

if INSTALL_DEPS:
    install_commands = [
        [
            sys.executable,
            '-m',
            'pip',
            'install',
            '-U',
            '--pre',
            'litert-torch-nightly',
            'litert-lm',
            'sentencepiece',
            'protobuf',
            'transformers',
            'huggingface_hub',
            'accelerate',
            'safetensors',
            'packaging',
        ],
        [
            sys.executable,
            '-m',
            'pip',
            'install',
            '-U',
            '--pre',
            'torch',
            'torchao',
            '--index-url',
            'https://download.pytorch.org/whl/nightly/cu128',
        ],
    ]
    for command in install_commands:
        record_event('install_start', command=' '.join(command))
        subprocess.run(command, check=True)
        record_event('install_done', command=' '.join(command))
    print('Dependency install finished. If torch or torchao changed, restart runtime, rerun Cell 1, then continue.')
    if AUTO_RESTART_AFTER_INSTALL:
        record_event('runtime_restart_requested')
        os.kill(os.getpid(), 9)
else:
    print('INSTALL_DEPS=0, skipping dependency install.')

## Exporter verification

This checks that the nightly exporter, Torch, and torchao are coherent before spending time on conversion.

In [ ]:
# Cell 3: Verify exporter imports and CLI availability.
import importlib
import importlib.util
from packaging.version import Version

import torch

record_event('exporter_verify_start', torch_version=torch.__version__)
print(f'torch={torch.__version__}')

if Version(torch.__version__.split('+')[0]) < Version('2.11.0.dev0'):
    print('WARNING: LiteRT Torch export is expected to need a recent nightly torch build.')

try:
    import torchao
    print(f'torchao={getattr(torchao, "__version__", "unknown")}')
    quantize_spec = importlib.util.find_spec('torchao.quantization.pt2e.quantize_pt2e')
    print(f'torchao.quantization.pt2e.quantize_pt2e available={quantize_spec is not None}')
except Exception as exc:
    raise RuntimeError(f'torchao import failed: {exc}') from exc

help_cmd = [sys.executable, '-m', 'litert_torch.generative.export_hf', '--help']
help_proc = subprocess.run(help_cmd, text=True, capture_output=True, check=False)
help_text = (help_proc.stdout or '') + (help_proc.stderr or '')
print('export_hf --help returncode:', help_proc.returncode)
print(help_text[:4000])

if 'Please upgrade to torch >= 2.11.0' in help_text:
    raise RuntimeError('litert_torch still sees an old torch. Restart runtime after install.')
if "No module named 'torchao.quantization.pt2e.quantize_pt2e'" in help_text:
    raise RuntimeError('litert_torch still sees incompatible torchao. Reinstall torchao from nightly index and restart.')

required_flags = ['--prefill_lengths', '--cache_length', '--quantization_recipe']
missing_flags = [flag for flag in required_flags if flag not in help_text]
if missing_flags:
    raise RuntimeError(f'LiteRT exporter help is missing expected flags: {missing_flags}')

help_looks_valid = (
    'SYNOPSIS' in help_text
    and 'POSITIONAL ARGUMENTS' in help_text
    and not missing_flags
)
if help_proc.returncode != 0 and help_looks_valid:
    print('Valid Fire help with non-zero return code; continuing is OK.')
elif help_proc.returncode != 0:
    raise RuntimeError('LiteRT exporter is not importable. See help output above.')

record_event('exporter_verify_done', torch_version=torch.__version__)

## Prompt budget audit

The v5 E2B router prompts are close to the 2048-token context budget. This audit is approximate (`chars / 4`) but useful enough to catch accidental prompt growth before conversion.

In [ ]:
# Cell 4: Approximate input-token budget audit for the v5 E2B router dataset.
import statistics

REPO_DIR = Path(os.environ.get('GEMMAFIT_REPO_DIR', '/content/GemmaFit'))
DATASET_PATH = Path(os.environ.get(
    'GEMMAFIT_ROUTER_DATASET',
    str(REPO_DIR / 'finetune' / 'data' / 'gemmafit_v5_2_e2b_evidence_router.json'),
))
CLONE_REPO_FOR_AUDIT = os.environ.get('CLONE_REPO_FOR_AUDIT', '0') == '1'
REPO_URL = os.environ.get('GEMMAFIT_REPO_URL', 'https://github.com/bkttt0429/GemmaFit.git')
REPO_BRANCH = os.environ.get('GEMMAFIT_REPO_BRANCH', 'codex/e2b-video-capability-probes')

if not DATASET_PATH.exists() and CLONE_REPO_FOR_AUDIT:
    if REPO_DIR.exists():
        subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)


def extract_prompt_text(row: Any) -> str:
    if isinstance(row, dict):
        for key in ('prompt', 'input', 'text'):
            value = row.get(key)
            if isinstance(value, str) and value.strip():
                return value
        messages = row.get('messages') or row.get('conversations')
        if isinstance(messages, list):
            parts = []
            for message in messages:
                if isinstance(message, dict):
                    content = message.get('content') or message.get('value') or message.get('text')
                    if isinstance(content, str):
                        parts.append(content)
            if parts:
                return '\n'.join(parts)
    return json.dumps(row, ensure_ascii=False)


def percentile(sorted_values: list[float], pct: float) -> float:
    if not sorted_values:
        raise ValueError('empty values')
    index = (len(sorted_values) - 1) * pct / 100.0
    lower = int(index)
    upper = min(lower + 1, len(sorted_values) - 1)
    weight = index - lower
    return sorted_values[lower] * (1.0 - weight) + sorted_values[upper] * weight

if DATASET_PATH.exists():
    payload = json.loads(DATASET_PATH.read_text(encoding='utf-8'))
    rows = payload if isinstance(payload, list) else payload.get('rows') or payload.get('data') or payload.get('examples')
    if not isinstance(rows, list):
        raise TypeError(f'Unsupported dataset shape in {DATASET_PATH}')
    approx_tokens = sorted(max(1, len(extract_prompt_text(row)) // 4) for row in rows)
    stats = {
        'dataset_path': str(DATASET_PATH),
        'rows': len(approx_tokens),
        'approx_chars_per_token': 4,
        'p50': round(percentile(approx_tokens, 50), 1),
        'p90': round(percentile(approx_tokens, 90), 1),
        'p95': round(percentile(approx_tokens, 95), 1),
        'p99': round(percentile(approx_tokens, 99), 1),
        'max': max(approx_tokens),
    }
    print(json.dumps(stats, indent=2, sort_keys=True))
    for name in RUN_VARIANTS:
        cache_length = VARIANT_BY_NAME[name]['cache_length']
        if stats['max'] > cache_length:
            print(f'WARNING: approx max prompt tokens {stats["max"]} exceeds cache_length={cache_length} for {name}')
        elif stats['p95'] > cache_length * 0.9:
            print(f'WARNING: p95 prompt tokens {stats["p95"]} is close to cache_length={cache_length} for {name}')
    record_event('prompt_budget_audit_done', **stats)
else:
    print(f'Dataset not found, skipping prompt audit: {DATASET_PATH}')
    print('Set CLONE_REPO_FOR_AUDIT=1 or GEMMAFIT_ROUTER_DATASET=/path/to/json to enable it.')
    record_event('prompt_budget_audit_skipped', dataset_path=str(DATASET_PATH))

## Convert matrix

Each variant exports to local `/content` first, then copies the `.litertlm`, exporter directory, log, hash, and state back to Drive. Existing artifacts are reused unless `FORCE_EXPORT=1`.

In [ ]:
# Cell 5: Run LiteRT-LM conversion variants.

def run_streaming(command: list[str], log_path: Path, cwd: Path | None = None) -> int:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('a', encoding='utf-8', errors='replace') as log_handle:
        log_handle.write(f'\n===== START {now_iso()} =====\n')
        log_handle.write('COMMAND: ' + ' '.join(command) + '\n')
        log_handle.flush()
        process = subprocess.Popen(
            command,
            cwd=str(cwd) if cwd is not None else None,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='')
            log_handle.write(line)
        rc = process.wait()
        log_handle.write(f'===== END {now_iso()} rc={rc} =====\n')
        return rc


def convert_one_variant(variant: dict[str, Any]) -> dict[str, Any]:
    name = variant['name']
    final_artifact = variant_artifact_path(name)
    drive_export_dir = variant_export_dir(name)
    log_path = variant_log_path(name)

    if final_artifact.exists() and final_artifact.stat().st_size > 0 and not FORCE_EXPORT:
        record_event('variant_skip_existing', variant=name, artifact=str(final_artifact), bytes=final_artifact.stat().st_size)
        return {
            'variant': name,
            'status': 'skipped_existing',
            'success': True,
            'artifact': str(final_artifact),
            'bytes': final_artifact.stat().st_size,
            'sha256': file_sha256(final_artifact),
            'log_path': str(log_path),
            'export_dir': str(drive_export_dir),
            **variant,
        }

    local_dir = LOCAL_WORK_ROOT / name
    if local_dir.exists():
        shutil.rmtree(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)

    command = [
        sys.executable,
        '-m',
        'litert_torch.generative.export_hf',
        str(MERGED_PATH),
        str(local_dir),
        f'--prefill_lengths={prefill_arg(variant["prefill_lengths"])}',
        f'--cache_length={variant["cache_length"]}',
        f'--quantization_recipe={variant["quantization_recipe"]}',
        '--externalize_embedder',
        '--use_jinja_template',
    ]
    if variant.get('experimental_lightweight_conversion'):
        command.append('--experimental_lightweight_conversion')

    started = time.time()
    record_event('variant_export_start', variant=name, command=' '.join(command), log_path=str(log_path))
    rc = run_streaming(command, log_path)
    elapsed_sec = round(time.time() - started, 3)

    result: dict[str, Any] = {
        'variant': name,
        'return_code': rc,
        'elapsed_sec': elapsed_sec,
        'log_path': str(log_path),
        'export_dir': str(drive_export_dir),
        **variant,
    }
    if rc != 0:
        result.update({'status': 'failed_export', 'success': False})
        record_event('variant_export_failed', variant=name, return_code=rc, elapsed_sec=elapsed_sec, log_path=str(log_path))
        return result

    try:
        litertlm = find_litertlm(local_dir)
        if drive_export_dir.exists():
            shutil.rmtree(drive_export_dir)
        shutil.copytree(local_dir, drive_export_dir)
        shutil.copy2(litertlm, final_artifact)
        result.update({
            'status': 'converted',
            'success': True,
            'artifact': str(final_artifact),
            'bytes': final_artifact.stat().st_size,
            'sha256': file_sha256(final_artifact),
        })
        if FORCE_FINAL or not LITERTLM_FINAL.exists() or LITERTLM_FINAL.stat().st_size == 0:
            shutil.copy2(final_artifact, LITERTLM_FINAL)
            result['copied_to_canonical'] = str(LITERTLM_FINAL)
        record_event(
            'variant_export_done',
            variant=name,
            elapsed_sec=elapsed_sec,
            artifact=str(final_artifact),
            bytes=result['bytes'],
            sha256=result['sha256'],
        )
        return result
    except Exception as exc:
        result.update({'status': 'failed_artifact_copy', 'success': False, 'error': repr(exc)})
        record_event('variant_artifact_copy_failed', variant=name, error=repr(exc))
        return result

results: list[dict[str, Any]] = []
for variant_name in RUN_VARIANTS:
    result = convert_one_variant(VARIANT_BY_NAME[variant_name])
    results.append(result)
    write_json(MATRIX_SUMMARY, {
        'run_name': RUN_NAME,
        'merged_path': str(MERGED_PATH),
        'artifact_prefix': ARTIFACT_PREFIX,
        'litertlm_final': str(LITERTLM_FINAL),
        'results': results,
        'summary_ts': now_iso(),
    })
    if result.get('success') and STOP_AFTER_FIRST_SUCCESS:
        record_event('matrix_stop_after_first_success', variant=variant_name)
        break

successes = [item for item in results if item.get('success')]
failures = [item for item in results if not item.get('success')]
print(json.dumps({'successes': successes, 'failures': failures}, indent=2, sort_keys=True))
if not successes:
    raise RuntimeError('No LiteRT-LM variant converted successfully. Open the variant logs under EXPORT_MATRIX_DIR.')
record_event('matrix_done', success_count=len(successes), failure_count=len(failures), summary=str(MATRIX_SUMMARY))

## Package local handoff bundles

These bundles are meant to be downloaded to the Windows repo and finalized with `finetune/prepare_litert_artifact.py`. The `.litertlm` inside each bundle is stored under the canonical Android resolver name.

In [ ]:
# Cell 6: Package converted variants for local Windows finalize.

def safe_zip_write_json(zip_handle: zipfile.ZipFile, arcname: str, payload: dict[str, Any]) -> None:
    zip_handle.writestr(arcname, json.dumps(payload, indent=2, sort_keys=True) + '\n')


def make_done_marker(result: dict[str, Any]) -> dict[str, Any]:
    return {
        'model': RUN_NAME,
        'profile': 'v5-e2b',
        'litertlm_path': f'models/{ARTIFACT_PREFIX}.litertlm',
        'source_litertlm': result.get('artifact'),
        'litert_variant': result.get('variant'),
        'conversion_status': 'converted_unverified',
        'conversion_summary': {
            'quantization_recipe': result.get('quantization_recipe'),
            'cache_length': result.get('cache_length'),
            'prefill_lengths': result.get('prefill_lengths'),
            'bytes': result.get('bytes'),
            'sha256': result.get('sha256'),
            'note': result.get('note'),
        },
        'eval_summary': {
            'selected_rows': 1000,
            'json_parse_rate': 0.982,
            'empty_or_unparseable_outputs': 18,
            'gate_failures': [],
            'hard_case_pass_rate': 0.997,
            'forbidden_claim_rate': 0.0,
        },
        'created_at_utc': now_iso(),
    }

summary_payload = json.loads(MATRIX_SUMMARY.read_text(encoding='utf-8'))
successes = [item for item in summary_payload.get('results', []) if item.get('success')]
if not successes:
    raise RuntimeError('No successful matrix result to package.')

bundle_rows = []
for result in successes:
    artifact = Path(result['artifact'])
    if not artifact.exists():
        raise FileNotFoundError(f'Converted artifact missing: {artifact}')
    name = result['variant']
    out_zip = bundle_path(name)
    with zipfile.ZipFile(out_zip, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as bundle:
        bundle.write(artifact, f'models/{ARTIFACT_PREFIX}.litertlm')
        safe_zip_write_json(bundle, 'finetune/metrics/training_done_v5_e2b.json', make_done_marker(result))
        safe_zip_write_json(bundle, 'finetune/metrics/litert_matrix_v5_e2b.json', summary_payload)
        for source_path, arcname in [
            (RUN_STATE, 'finetune/metrics/run_state_v5_e2b_litert_matrix.json'),
            (RUN_EVENTS, 'finetune/metrics/run_events_v5_e2b_litert_matrix.jsonl'),
            (DISCONNECT_POINTS, 'finetune/metrics/disconnect_points_v5_e2b_litert_matrix.jsonl'),
            (Path(result.get('log_path', '')), f'finetune/metrics/{ARTIFACT_PREFIX}-{name}-export.log'),
        ]:
            if source_path.exists():
                bundle.write(source_path, arcname)
    row = {
        'variant': name,
        'bundle': str(out_zip),
        'bundle_bytes': out_zip.stat().st_size,
        'artifact_bytes': artifact.stat().st_size,
        'artifact_sha256': result.get('sha256'),
    }
    bundle_rows.append(row)
    record_event('bundle_created', **row)

print(json.dumps(bundle_rows, indent=2, sort_keys=True))
print('\nWindows finalize commands after downloading one bundle to D:\\GemmaFit:')
for row in bundle_rows:
    bundle_name = Path(row['bundle']).name
    print(f'python finetune\\prepare_litert_artifact.py --profile v5-e2b --source-bundle "D:\\GemmaFit\\{bundle_name}" --run-smoke')
print('\nIf local LiteRT smoke dependencies are unavailable, rerun without --run-smoke and do Pixel smoke next.')

## Pixel smoke after local finalize

After downloading one bundle to `D:\GemmaFit` and running `prepare_litert_artifact.py`, push the canonical model name expected by Android:

```powershell
adb push models\gemmafit-v5-e2b-evidence-router.litertlm /sdcard/Android/data/com.gemmafit/files/gemmafit-v5-e2b-evidence-router.litertlm
adb shell am start -n com.gemmafit/.MainActivity
adb shell "timeout 90 content read --uri content://com.gemmafit.debug/litert_smoke" > debug_litert_smoke_v5_e2b.json
adb logcat -d -v time > debug_litert_smoke_v5_e2b_logcat.txt
```

Pass criteria:
- Backend name starts with `litert-lm`.
- Smoke result reports `success=true`.
- The returned tool name is one of the v5 E2B allowed tools.
- `evidence_refs` are present and valid.

Failure triage:
- `engine_initialize` fails on both GPU and CPU: treat as conversion/runtime artifact issue and try the next matrix variant.
- `litert_lm_no_valid_tool_call`: the engine is running; debug prompt/schema/model behavior rather than conversion.
- Model missing or wrong source: verify Android resolver path and sideload destination.